In [2]:
import pandas as pd
import sqlite3
pd.set_option('display.max_rows', None)
conn = sqlite3.connect('itf_tournaments.db')
curs = conn.cursor()
curs.execute("PRAGMA foreign_keys=ON;")
#conn.close()

In [ ]:
query1 = pd.read_sql("""
                    
                WITH tWTN AS (
                SELECT
                        *
                FROM tTournaments
                WHERE date_started LIKE '%21 Apr%' 
                     OR date_started LIKE '%28 Apr%' 
                     OR date_started LIKE '%May%'
                )
                
                SELECT qualysize, count(qualysize)
                FROM tWTN
                WHERE qualybyes != 0
                GROUP BY qualysize
                ORDER BY qualysize desc
            
            ;""", conn)

query2 = pd.read_sql("""
        SELECT *
        FROM tPlayerInfo
        WHERE tournament_key = 'm-itf-esp-2025-026'
            
;""", conn)

query3 = pd.read_sql("""
        SELECT *
        FROM tTournaments
        WHERE country = 'SPAIN'
        --AND date_started LIKE '%2025%'
        --AND city NOT LIKE '%FL%'
        --AND city NOT LIKE '% CA%'
;""", conn)

query3

In [125]:
# Return the last player in for each tournament
df = pd.read_sql("""
                     WITH NonByes AS (
                     SELECT tournament_key
                     FROM tTournaments
                     WHERE date_started = '08 Sep 2025'
                     AND qualybyes == '0'
                     ),

                    RankedPlayers AS (
                        SELECT *,
                            ROW_NUMBER() OVER (
                                PARTITION BY tournament_key
                                ORDER BY acceptancelist_type ASC, acceptancelist_number DESC
                            ) AS row_num
                        FROM tPlayerInfo
                        WHERE tournament_key IN (SELECT tournament_key FROM NonByes)
                        AND (designation = 'DA' OR designation = '(A)')
                    )
                    SELECT rank_type, rank_value, tournament_key
                    FROM RankedPlayers
                    WHERE row_num = 1
                    ORDER BY rank_type DESC, rank_value DESC


                    
            ;""", conn)
df

,rank_type,rank_value,tournament_key
0,NONE,0,m-itf-aus-2025-007
1,NONE,0,m-itf-esp-2025-047
2,NONE,0,m-itf-ina-2025-007
3,NONE,0,m-itf-ita-2025-022
4,NATIONAL,449,m-itf-fra-2025-025
5,ITF,1749,m-itf-jpn-2025-019


In [108]:
def get_user_ranking():
    ranking_types = ["ATP", "ITF", "WTN", "NATIONAL"]
    for rtype in ranking_types:
        while True:
            answer = input(f"Do you have a {rtype} ranking? (y/n): ").strip().lower()
            if answer == "y":
                while True:
                    value = input(f"Enter your {rtype} ranking number: ")
                    if rtype == "WTN":
                        try:
                            float_value = float(value)
                            return rtype, float_value
                        except ValueError:
                            print("Please enter a valid number (integer or decimal) for WTN. Try again.")
                    else:
                        if value.isdigit():
                            return rtype, int(value)
                        else:
                            print("Please enter an integer ranking number. Try again.")
            elif answer == "n":
                break
            else:
                print("Please enter 'y' or 'n'. Try again.")
    print("No ranking provided.")
    return None, None

# Example usage:
rtype, rval = get_user_ranking()
print(f"Ranking type: {rtype}, Ranking value: {rval}")

Please enter 'y' or 'n'. Try again.
Ranking type: WTN, Ranking value: 19.3


In [ ]:
import pandas as pd

rank_order = {"ATP": 4, "ITF": 3, "WTN": 2, "NATIONAL": 1, None: 0}

def normalize_rank_type(x):
    """Turn strings like 'NONE', 'null', '' or NaN into Python None; otherwise return uppercased string."""
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s in ("NONE", "NULL", "NAN", ""):
        return None
    return s

# normalize user's rank type once
user_rank_type = normalize_rank_type(rtype)   # rtype is the user's rank type (e.g. "ATP", "NATIONAL", "NONE", or None)
user_rank = rank_order.get(user_rank_type, 0)

filtered = []
for idx, row in df.iterrows():
    tour_rank_type = normalize_rank_type(row.get('rank_type'))
    tour_rank_value = row.get('rank_value')
    tour_rank = rank_order.get(tour_rank_type, 0)

    # Case: user is NATIONAL -> include tournaments where last player was NATIONAL or None
    if user_rank_type == "NATIONAL":
        if tour_rank_type in ("NATIONAL", None):
            filtered.append(row)
        continue

    # Case: user is unranked / None -> include tournaments where last player was None
    if user_rank_type is None:
        if tour_rank_type is None:
            filtered.append(row)
        continue

    # Case: ATP / ITF / WTN -> keep original comparison logic
    if user_rank > tour_rank:
        filtered.append(row)
    elif user_rank == tour_rank:
        # compare rank values (rval is the user's numeric ranking value)
        try:
            if pd.isna(rval) or pd.isna(tour_rank_value):
                # if either value is missing, skip this row (can't compare)
                continue
            if float(rval) <= float(tour_rank_value):
                filtered.append(row)
        except Exception:
            # if conversion to float fails, skip this row
            continue
    else:
        # user_rank < tour_rank -> cannot get in
        continue

# Show tournaments you would get into
if filtered:
    result_df = pd.DataFrame(filtered)
    print(result_df[['tournament_key', 'rank_type', 'rank_value']])
else:
    print("No tournaments found that match your ranking.")



       tournament_key rank_type rank_value
0  m-itf-aus-2025-007      NONE          0
1  m-itf-esp-2025-047      NONE          0
2  m-itf-ina-2025-007      NONE          0
3  m-itf-ita-2025-022      NONE          0
4  m-itf-fra-2025-025  NATIONAL        449


In [121]:
result_df

,rank_type,rank_value,tournament_key
0,WTN,30.37,m-itf-srb-2025-024
1,WTN,19.47,m-itf-esp-2025-045
3,NONE,0,m-itf-ita-2025-036
4,NONE,0,m-itf-tur-2025-028
5,NATIONAL,93,m-itf-hun-2025-008


In [131]:
all_touraments = pd.read_sql("""

SELECT tournament_key, city, country, date_started, qualysize
FROM tTournaments
WHERE date_started = '08 Sep 2025'

""", conn)

combined = pd.merge(
    all_touraments,        # left DataFrame
    result_df,             # right DataFrame (your filtered tournaments)
    on='tournament_key',   # key column to match on
    how='inner'            # inner join = only tournaments in both
)

combined

,tournament_key,city,country,date_started,qualysize,rank_type,rank_value
0,m-itf-ina-2025-007,BALI,INDONESIA,08 Sep 2025,32,NONE,0
1,m-itf-fra-2025-025,PLAISIR,FRANCE,08 Sep 2025,48,NATIONAL,449
2,m-itf-aus-2025-007,TAMWORTH,AUSTRALIA,08 Sep 2025,48,NONE,0
3,m-itf-esp-2025-047,MADRID,SPAIN,08 Sep 2025,64,NONE,0
4,m-itf-ita-2025-022,POZZUOLI,ITALY,08 Sep 2025,32,NONE,0


In [135]:
all_touraments = pd.read_sql("""

SELECT DISTINCT(date_started)
FROM tTournaments
-- WHERE date_started = '08 Sep 2025'

""", conn)



In [139]:
from datetime import datetime

def get_date():
    """Ask the user for a date in 'DD Mon YYYY' format (e.g., '03 Sep 2025') and validate it."""
    while True:
        date_input = input("Enter the tournament start date (e.g., 03 Sep 2025): ").strip()
        try:
            # Parse with day, abbreviated month name, and year
            valid_date = datetime.strptime(date_input, "%d %b %Y")
            # Return formatted date string to match your DB format
            return valid_date.strftime("%d %b %Y")
        except ValueError:
            print("Invalid format. Please enter the date as 'DD Mon YYYY' (e.g., 03 Sep 2025).")


get_date()

Invalid format. Please enter the date as 'DD Mon YYYY' (e.g., 03 Sep 2025).


'04 Nov 2024'